In [1]:
from resources.imports import *

import torch
import torch.nn as nn

from resources.MLdata import DATA
from resources.MLfunc import EarlyStopping, CombinedCurveLoss, QuantileLoss, QuantileLossMATLAB
from resources.MLfunc import hOpt_model, hOpt_compare, hOpt_best_summary
from resources.MLmodels import *

In [2]:
%load_ext autoreload
%autoreload 2

# Stress-Strain Curve

## Multi-Layer Perceptrion (MLP)

### DATA

In [3]:
DAT_MLP = DATA(
    path=1, 
    path_add="",
    load=True, 
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC", 
    dis="disNodes", 
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="MLP",
    scale=("symm", "inout"),
    reduce_dim=False, #("PCA", "out", 0.95, 10, True)
    round_decimals=5
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
DAT = DAT_MLP
if DAT.UTmechTest:
    print("UT shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("Train -  in (min, max):", DAT.UT_train_in.min(), DAT.UT_train_in.max(), "\n        out (min, max):", DAT.UT_train_out.min(), DAT.UT_train_out.max())
    print("  Val -  in (min, max):", DAT.UT_val_in.min(), DAT.UT_val_in.max(), "\n        out (min, max):", DAT.UT_val_out.min(), DAT.UT_val_out.max())
    print(" Test -  in (min, max):", DAT.UT_test_in.min(), DAT.UT_test_in.max(), "\n        out (min, max):", DAT.UT_test_out.min(), DAT.UT_test_out.max())
    if DAT.FTmechTest: print("\n========================================================\n")
if DAT.FTmechTest:
    print("FT shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("Train -  in (min, max):", DAT.FT_train_in.min(), DAT.FT_train_in.max(), "\n        out (min, max):", DAT.FT_train_out.min(), DAT.FT_train_out.max())
    print("  Val -  in (min, max):", DAT.FT_val_in.min(), DAT.FT_val_in.max(), "\n        out (min, max):", DAT.FT_val_out.min(), DAT.FT_val_out.max())
    print(" Test -  in (min, max):", DAT.FT_test_in.min(), DAT.FT_test_in.max(), "\n        out (min, max):", DAT.FT_test_out.min(), DAT.FT_test_out.max())

UT shape: (5561, 1444) (5561, 201)
Train -  in (min, max): -1.0 1.0 
        out (min, max): -1.0 1.0
  Val -  in (min, max): -1.0 1.0 
        out (min, max): -1.6191947669483455 1.103937947512592
 Test -  in (min, max): -1.0 1.0 
        out (min, max): -1.595077081066206 1.0862485225493743


### HPO

In [5]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Full HPO run on the full DATA split. Re-running resumes the saved Optuna study.

MLP_hOpt_model_space = {
    "depth": [2, 3, 4, 5],
    "width": [128, 256, 512],
    "block": ["mlp", "res"],
    "act": ["relu", "gelu", "mish"],
    "norm": [None, "batch", "layer"],
    "dropout": {"type": "float", "low": 0.0, "high": 0.25},
    "head_norm": [None, "layer"],
    "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
}

MLP_hOpt_loss_space = {
    "family": ["mse", "combined"],
    "mse_weight": [0.25, 0.5, 1.0],
    "weighted_mse_weight": [0.0, 0.5, 1.0, 2.0],
    "derivative_weight": [0.0, 0.02, 0.05, 0.10],
    "peak_weight": [0.0, 0.10, 0.25, 0.50],
    "energy_weight": [0.0, 0.05, 0.10, 0.25],
    "peak_location_weight": [0.0, 0.02, 0.05],
    "reduction": ["mean"],
    "derivative_order": [1],
    "SoftPeak_beta": [10.0, 20.0, 40.0],
    "UT": {
        "zone_boundaries": (67, 134),
        "zone_weights": (1.0, 5.0, 2.0),
    },
}

MLP_hOpt_train_space = {
    "optimizer": ["adamw"],
    "lr": {"type": "float", "low": 5e-5, "high": 2e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-8, "high": 1e-3, "log": True},
    "batch": [16, 32, 64],
    "n_epochs": {"type": "fixed", "value": 250},
    "metric": ["rmse"],
    "scheduler": ["plateau"],
    "scheduler_factor": [0.3, 0.5],
    "scheduler_patience": [10, 20],
    "scheduler_threshold": {"type": "fixed", "value": 1e-4},
    "early_stop": [True],
    "early_stop_patience": [30, 50],
    "early_stop_min_delta": {"type": "fixed", "value": 1e-5},
    "verbose": {"type": "fixed", "value": 0},
}

study_MLP_full = hOpt_model(
    typ="mlp",
    data=DAT_MLP,
    n_trials=50,
    model_space=MLP_hOpt_model_space,
    loss_space=MLP_hOpt_loss_space,
    train_space=MLP_hOpt_train_space,
    seed=42,
    device=device,
    save=True,
    name="MLP_full_hOpt",
    n_jobs=1,
    show_progress_bar=True,
)

hOpt_best_summary(study_MLP_full)

c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2269: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-08 14:00:39,347] Using an existing study with name 'MLP_full_hOpt' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

 -> Epoch 1/250 || LOSS - train: 1.534778, val: 0.543991 | MSE - train: 1.534778, val: 0.543991 | RMSE - train: 1.238862, val: 0.737558 | LR: 5.70e-05
================ Training Complete ================
 Best Epoch: 58, with LOSS: 0.086816, MSE: 0.086816 and RMSE: 0.294645 ==================
[I 2026-05-08 14:01:26,575] Trial 50 finished with value: 0.29464515849660217 and parameters: {'mlp_act': 'relu', 'mlp_norm': None, 'mlp_dropout': 0.13244784101936152, 'mlp_head_dropout': 0.11374865809015285, 'mlp_depth': 3, 'mlp_width': 128, 'mlp_block': 'res', 'mlp_head_norm': 'layer', 'loss_family': 'mse', 'optimizer': 'adamw', 'lr': 5.701428153379758e-05, 'weight_decay': 0.0002874262257345196, 'batch': 64, 'objective_metric': 'rmse', 'w_init': 'auto', 'scheduler': 'plateau', 'scheduler_factor': 0.5, 'scheduler_patience': 20, 'early_stop': True, 'early_stop_patience': 30}. Best is trial 47 with value: 0.2939155574299007.
 -> Epoch 1/250 || LOSS - train: 1.379141, val: 0.288845 | MSE - train: 1.3

{'best_value': 0.2938914024458132,
 'best_params': {'mlp_act': 'gelu',
  'mlp_norm': None,
  'mlp_dropout': 0.24575130097402698,
  'mlp_head_dropout': 0.16385019417035762,
  'mlp_depth': 4,
  'mlp_width': 128,
  'mlp_block': 'res',
  'mlp_head_norm': 'layer',
  'loss_family': 'mse',
  'optimizer': 'adamw',
  'lr': 8.895206317091309e-05,
  'weight_decay': 0.00012623890121858924,
  'batch': 64,
  'objective_metric': 'rmse',
  'w_init': 'auto',
  'scheduler': 'plateau',
  'scheduler_factor': 0.5,
  'scheduler_patience': 20,
  'early_stop': True,
  'early_stop_patience': 30},
 'best_user_attrs': {'loss_params': {'family': 'mse'},
  'model_params': {'in_size': 1444,
   'out_size': 201,
   'h_size': [128, 128, 128, 128],
   'act': 'gelu',
   'block': 'res',
   'norm': None,
   'dropout': 0.24575130097402698,
   'head_norm': 'layer',
   'head_dropout': 0.16385019417035762},
  'task_scores': [0.2938914024458132],
  'train_params': {'lr': 8.895206317091309e-05,
   'weight_decay': 0.000126238901

### MODEL

In [ ]:
MLP1 = MODEL(
    typ=DAT.model,
    model=MLP(in_size=DAT.UT_train_in.shape[-1], 
              h_size=[4096, 2048, 1024, 1024, 1024, 512, 512], 
              out_size=DAT.UT_train_out.shape[-1], 
              act="relu",
              block="mlp",
              norm="batch", 
              dropout=0.0).to(device), 
    lossf=QuantileLossMATLAB(quantiles=[0.5, 0.45, 0.1], zone_boundaries=(67, 134), err_type="L2"), #nn.MSELoss(reduction="mean"), #CustomQuantileLoss(quantiles=[0.5, 0.45, 0.1], zone_boundaries=(50, 150), err_type="L2"),
    opt=("adam", 6.016e-6), #("adam", 1e-4),
    batch=32,
    lr=9e-4,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.2427, 100, 1e-4), #("min", 0.2427, 18, 1e-4), 
    earlyStop=EarlyStopping(patience=500, min_delta=1e-4, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True
)

MLP1.summary()

In [ ]:
MLP1.train(n_epochs=1000, verbose=20, plot=True)    

In [ ]:
MLP1.save(path=None, name=None)

In [ ]:
MLP1.predict(test_dataloader=None, plot=True)

In [ ]:
for i in range(len(MLP1.UT_test_outputs)):
    plot_predictions(DAT.UT_OUT_df, MLP1.UT_test_outputs, MLP1.UT_truth, indx=i, d_out=False)

## Graph Neural Network (GNN)

### DATA

In [6]:
DAT_GNN = DATA(
    path=1, 
    path_add="",
    load=True, 
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC", 
    dis="disNodes", 
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="GNN",
    scale=("symm", "inout"),
    reduce_dim=False, #("PCA", "out", 0.95, 10, True)
    round_decimals=5,
    tr_params={"geom_feats": True, "coord_norm": True},
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [7]:
DAT = DAT_GNN
if DAT.UTmechTest:
    print("UT shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("Train -  in (min, max):", DAT.UT_train_in.min(), DAT.UT_train_in.max(), "\n        out (min, max):", DAT.UT_train_out.min(), DAT.UT_train_out.max())
    print("  Val -  in (min, max):", DAT.UT_val_in.min(), DAT.UT_val_in.max(), "\n        out (min, max):", DAT.UT_val_out.min(), DAT.UT_val_out.max())
    print(" Test -  in (min, max):", DAT.UT_test_in.min(), DAT.UT_test_in.max(), "\n        out (min, max):", DAT.UT_test_out.min(), DAT.UT_test_out.max())
    if DAT.FTmechTest: print("\n========================================================\n")
if DAT.FTmechTest:
    print("FT shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("Train -  in (min, max):", DAT.FT_train_in.min(), DAT.FT_train_in.max(), "\n        out (min, max):", DAT.FT_train_out.min(), DAT.FT_train_out.max())
    print("  Val -  in (min, max):", DAT.FT_val_in.min(), DAT.FT_val_in.max(), "\n        out (min, max):", DAT.FT_val_out.min(), DAT.FT_val_out.max())
    print(" Test -  in (min, max):", DAT.FT_test_in.min(), DAT.FT_test_in.max(), "\n        out (min, max):", DAT.FT_test_out.min(), DAT.FT_test_out.max())

UT shape: (5561, 800, 8) (5561, 201)
Train -  in (min, max): -1.0 1.0 
        out (min, max): -1.0 1.0
  Val -  in (min, max): -1.0 1.0 
        out (min, max): -1.6191947669483455 1.103937947512592
 Test -  in (min, max): -1.0 1.0 
        out (min, max): -1.595077081066206 1.0862485225493743


### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Full GCN/GAT HPO run on the full graph DATA split. Re-running resumes the saved Optuna studies.

GNN_hOpt_model_space = {
    "gcn": {
        "depth": [2, 3, 4],
        "width": [64, 128, 256],
        "act": ["relu", "gelu", "mish"],
        "norm": [None, "layer"],
        "dropout": {"type": "float", "low": 0.0, "high": 0.25},
        "head_norm": [None, "layer"],
        "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
        "pool": ["mean"],
    },
    "gat": {
        "depth": [1, 2, 3],
        "width": [32, 64, 128],
        "heads": [1, 2, 4],
        "act": ["relu", "gelu"],
        "norm": [None, "layer"],
        "dropout": {"type": "float", "low": 0.0, "high": 0.25},
        "att_dropout": {"type": "float", "low": 0.0, "high": 0.20},
        "head_norm": [None, "layer"],
        "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
        "pool": ["mean"],
    },
}

GNN_hOpt_loss_space = {
    "family": ["mse", "combined"],
    "mse_weight": [0.25, 0.5, 1.0],
    "weighted_mse_weight": [0.0, 0.5, 1.0, 2.0],
    "derivative_weight": [0.0, 0.02, 0.05, 0.10],
    "peak_weight": [0.0, 0.10, 0.25, 0.50],
    "energy_weight": [0.0, 0.05, 0.10, 0.25],
    "peak_location_weight": [0.0, 0.02, 0.05],
    "reduction": ["mean"],
    "derivative_order": [1],
    "SoftPeak_beta": [10.0, 20.0, 40.0],
    "UT": {
        "zone_boundaries": (67, 134),
        "zone_weights": (1.0, 5.0, 2.0),
    },
}

GNN_hOpt_train_space = {
    "optimizer": ["adamw"],
    "lr": {"type": "float", "low": 3e-5, "high": 1e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-8, "high": 1e-3, "log": True},
    "batch": [4, 8, 16],
    "n_epochs": {"type": "fixed", "value": 150},
    "metric": ["rmse"],
    "scheduler": ["plateau"],
    "scheduler_factor": [0.3, 0.5],
    "scheduler_patience": [8, 15],
    "scheduler_threshold": {"type": "fixed", "value": 1e-4},
    "early_stop": [True],
    "early_stop_patience": [25, 40],
    "early_stop_min_delta": {"type": "fixed", "value": 1e-5},
    "verbose": {"type": "fixed", "value": 0},
}

studies_GNN_full = hOpt_compare(
    typs=["gcn", "gat"],
    data=DAT_GNN,
    n_trials_per_typ=30,
    model_space=GNN_hOpt_model_space,
    loss_space=GNN_hOpt_loss_space,
    train_space=GNN_hOpt_train_space,
    seed=42,
    device=device,
    save=True,
    name="GNN_full_hOpt",
    n_jobs=1,
    show_progress_bar=True,
)

hOpt_best_summary(studies_GNN_full)

c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2269: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-08 15:40:00,305] A new study created in RDB with name: GNN_full_hOpt_gcn


  0%|          | 0/30 [00:00<?, ?it/s]

 -> Epoch 1/150 || LOSS - train: 0.116281, val: 0.105738 | MSE - train: 0.099301, val: 0.089095 | RMSE - train: 0.315121, val: 0.298487 | LR: 7.28e-04
================ Training Complete ================
 Best Epoch: 19, with LOSS: 0.104116, MSE: 0.086756 and RMSE: 0.294544 ==================
[I 2026-05-08 16:02:30,827] Trial 0 finished with value: 0.2945438155709612 and parameters: {'gcn_act': 'gelu', 'gcn_norm': None, 'gcn_dropout': 0.03899863008405066, 'gcn_head_dropout': 0.011616722433639893, 'gcn_depth': 2, 'gcn_width': 128, 'gcn_att_dropout': 0.07431868873739665, 'gcn_head_norm': 'layer', 'gcn_pool': 'mean', 'loss_family': 'combined', 'mse_weight': 1.0, 'weighted_mse_weight': 2.0, 'derivative_weight': 0.0, 'peak_weight': 0.1, 'energy_weight': 0.05, 'peak_location_weight': 0.02, 'loss_reduction': 'mean', 'loss_derivative_order': 1, 'loss_SoftPeak_beta': 20.0, 'optimizer': 'adamw', 'lr': 0.000727622244615796, 'weight_decay': 1.9674328025306071e-07, 'batch': 4, 'objective_metric': 'r

### MODEL

In [ ]:
GNN1 = MODEL(
    typ=DAT.model,
    model=GNN(in_size=DAT.UT_train_in.shape[-1], 
              h_size=[64, 64, 64, 64],
              out_size=DAT.UT_train_out.shape[-1], 
              act="relu",
              block="gat",
              norm="layer",
              dropout=0.2,
              bias=True,
              heads=2,
              pool="mean").to(device),
    lossf=nn.MSELoss(reduction="mean"),
    opt=("adam", 6.016e-6), #("adam", 1e-4),
    batch=36,
    lr=9e-4,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.2427, 100, 1e-4), 
    earlyStop=EarlyStopping(patience=150, min_delta=1e-4, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True
)

GNN1.summary()

In [ ]:
GNN1.train(n_epochs=1000, verbose=20, plot=True)

In [ ]:
GNN1.save(path=None, name=None)

In [ ]:
GNN1.predict(test_dataloader=None, plot=True)

In [ ]:
for i in range(len(GNN1.UT_test_outputs)):
    plot_predictions(DAT.UT_OUT_df, GNN1.UT_test_outputs, GNN1.UT_truth, indx=i, d_out=False)

## Transformer

### DATA

In [6]:
DAT_TR = DATA(
    path=1, 
    path_add="",
    load=True, 
    load_split=False,
    split_frac=0.9,
    split_seed=42,
    range_split=(True, False),
    save_split=False,
    LAT="FCC", 
    dis="disNodes", 
    dN=0.2,
    d_data="in",
    mechMode="UT",
    nsims=None,
    model="TR",
    scale=("symm", "inout"),
    reduce_dim=False, #("PCA", "out", 0.95, 10, True)
    round_decimals=5,
    tr_params={"geom_feats": True, "coord_norm": True}
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [7]:
DAT = DAT_TR
if DAT.UTmechTest:
    print("UT shape:", DAT.UT_train_in.shape, DAT.UT_train_out.shape)
    print("Train -  in (min, max):", DAT.UT_train_in.min(), DAT.UT_train_in.max(), "\n        out (min, max):", DAT.UT_train_out.min(), DAT.UT_train_out.max())
    print("  Val -  in (min, max):", DAT.UT_val_in.min(), DAT.UT_val_in.max(), "\n        out (min, max):", DAT.UT_val_out.min(), DAT.UT_val_out.max())
    print(" Test -  in (min, max):", DAT.UT_test_in.min(), DAT.UT_test_in.max(), "\n        out (min, max):", DAT.UT_test_out.min(), DAT.UT_test_out.max())
    if DAT.FTmechTest: print("\n========================================================\n")
if DAT.FTmechTest:
    print("FT shape:", DAT.FT_train_in.shape, DAT.FT_train_out.shape)
    print("Train -  in (min, max):", DAT.FT_train_in.min(), DAT.FT_train_in.max(), "\n        out (min, max):", DAT.FT_train_out.min(), DAT.FT_train_out.max())
    print("  Val -  in (min, max):", DAT.FT_val_in.min(), DAT.FT_val_in.max(), "\n        out (min, max):", DAT.FT_val_out.min(), DAT.FT_val_out.max())
    print(" Test -  in (min, max):", DAT.FT_test_in.min(), DAT.FT_test_in.max(), "\n        out (min, max):", DAT.FT_test_out.min(), DAT.FT_test_out.max())

UT shape: (5561, 800, 8) (5561, 201)
Train -  in (min, max): -1.0 1.0 
        out (min, max): -1.0 1.0
  Val -  in (min, max): -1.0 1.0 
        out (min, max): -1.6191947669483455 1.103937947512592
 Test -  in (min, max): -1.0 1.0 
        out (min, max): -1.595077081066206 1.0862485225493743


### HPO

In [ ]:
# HYPERPARAMETER OPTIMIZATION HERE.
# Full HPO run on the full Transformer DATA split. Re-running resumes the saved Optuna study.

TR_hOpt_model_space = {
    "d_model": [32, 64, 128],
    "n_heads": [1, 2, 4, 8],
    "n_layers": [1, 2, 3],
    "ff_mult": [2, 4],
    "head_depth": [0, 1, 2],
    "head_width": [64, 128, 256],
    "pool": ["mean", "cls"],
    "use_cls_token": [True, False],
    "act": ["relu", "gelu"],
    "encoder_act": ["gelu", "relu"],
    "block": ["mlp", "res"],
    "norm": ["layer"],
    "dropout": {"type": "float", "low": 0.0, "high": 0.25},
    "att_dropout": {"type": "float", "low": 0.0, "high": 0.20},
    "head_norm": ["same", None, "layer"],
    "head_dropout": {"type": "float", "low": 0.0, "high": 0.20},
    "pos_encoding": ["learned", "sinusoidal"],
}

TR_hOpt_loss_space = {
    "family": ["mse", "combined"],
    "mse_weight": [0.25, 0.5, 1.0],
    "weighted_mse_weight": [0.0, 0.5, 1.0, 2.0],
    "derivative_weight": [0.0, 0.02, 0.05, 0.10],
    "peak_weight": [0.0, 0.10, 0.25, 0.50],
    "energy_weight": [0.0, 0.05, 0.10, 0.25],
    "peak_location_weight": [0.0, 0.02, 0.05],
    "reduction": ["mean"],
    "derivative_order": [1],
    "SoftPeak_beta": [10.0, 20.0, 40.0],
    "UT": {
        "zone_boundaries": (67, 134),
        "zone_weights": (1.0, 5.0, 2.0),
    },
}

TR_hOpt_train_space = {
    "optimizer": ["adamw"],
    "lr": {"type": "float", "low": 3e-5, "high": 1e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-8, "high": 1e-3, "log": True},
    "batch": [4, 8, 16],
    "n_epochs": {"type": "fixed", "value": 150},
    "metric": ["rmse"],
    "scheduler": ["plateau"],
    "scheduler_factor": [0.3, 0.5],
    "scheduler_patience": [8, 15],
    "scheduler_threshold": {"type": "fixed", "value": 1e-4},
    "early_stop": [True],
    "early_stop_patience": [25, 40],
    "early_stop_min_delta": {"type": "fixed", "value": 1e-5},
    "verbose": {"type": "fixed", "value": 0},
}

study_TR_full = hOpt_model(
    typ="tr",
    data=DAT_TR,
    n_trials=35,
    model_space=TR_hOpt_model_space,
    loss_space=TR_hOpt_loss_space,
    train_space=TR_hOpt_train_space,
    seed=42,
    device=device,
    save=True,
    name="TR_full_hOpt",
    n_jobs=1,
    show_progress_bar=True,
)

hOpt_best_summary(study_TR_full)

c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Programs\Python39\lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  optuna_warn(
C:\Users\exy053\Documents\00-PhD-gitRepo\resources\MLfunc.py:2269: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner = optuna.pruners.PatientPruner(base_pruner, patience=5) if hasattr(optuna.pruners, "PatientPruner") else base_pruner
[I 2026-05-06 22:02:17,151] A new study created in RDB with name: TR_full_hOpt


  0%|          | 0/35 [00:00<?, ?it/s]

c:\Programs\Python39\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


 -> Epoch 1/150 || LOSS - train: 0.100586, val: 0.087133 | MSE - train: 0.100586, val: 0.087133 | RMSE - train: 0.317153, val: 0.295183 | LR: 3.30e-04
================ Training Complete ================
 Best Epoch: 47, with LOSS: 0.086385, MSE: 0.086385 and RMSE: 0.293913 ==================
[I 2026-05-07 02:05:06,101] Trial 0 finished with value: 0.2939125272185462 and parameters: {'tr_act': 'gelu', 'tr_norm': 'layer', 'tr_dropout': 0.18299848545285127, 'tr_head_dropout': 0.11973169683940732, 'tr_d_model': 32, 'tr_n_heads': 1, 'tr_pool': 'mean', 'tr_use_cls_token': True, 'tr_head_depth': 2, 'tr_head_width': 256, 'tr_n_layers': 3, 'tr_ff_mult': 4, 'tr_encoder_act': 'relu', 'tr_head_block': 'mlp', 'tr_att_dropout': 0.12150897038028768, 'tr_head_norm': 'layer', 'tr_pos_encoding': 'learned', 'loss_family': 'mse', 'optimizer': 'adamw', 'lr': 0.00033046478547966453, 'weight_decay': 1.5876781526923984e-06, 'batch': 8, 'objective_metric': 'rmse', 'w_init': 'auto', 'scheduler': 'plateau', 'sch

c:\Programs\Python39\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


 -> Epoch 1/150 || LOSS - train: 0.127361, val: 0.087175 | MSE - train: 0.127361, val: 0.087175 | RMSE - train: 0.356876, val: 0.295255 | LR: 2.75e-04
================ Training Complete ================
 Best Epoch: 56, with LOSS: 0.086394, MSE: 0.086394 and RMSE: 0.293929 ==================
[I 2026-05-07 13:08:05,202] Trial 2 finished with value: 0.29392902224254286 and parameters: {'tr_act': 'gelu', 'tr_norm': 'layer', 'tr_dropout': 0.10677694715656408, 'tr_head_dropout': 0.16360295318449863, 'tr_d_model': 32, 'tr_n_heads': 1, 'tr_pool': 'mean', 'tr_use_cls_token': False, 'tr_head_depth': 1, 'tr_head_width': 128, 'tr_n_layers': 3, 'tr_ff_mult': 2, 'tr_encoder_act': 'relu', 'tr_head_block': 'mlp', 'tr_att_dropout': 0.09789055205551261, 'tr_head_norm': 'same', 'tr_pos_encoding': 'learned', 'loss_family': 'mse', 'optimizer': 'adamw', 'lr': 0.00027545227564861415, 'weight_decay': 1.471121537912197e-05, 'batch': 16, 'objective_metric': 'rmse', 'w_init': 'auto', 'scheduler': 'plateau', 'sc

### MODEL

In [ ]:
TR1 = MODEL(
    typ=DAT.model,
    model=Transformer(
        in_size=DAT.UT_train_in.shape[-1] if DAT.UTmechTest else DAT.FT_train_in.shape[-1],
        seq_len=DAT.UT_train_in.shape[-2] if DAT.UTmechTest else DAT.FT_train_in.shape[-2],
        h_size=[64],
        out_size=DAT.UT_train_out.shape[-1] if DAT.UTmechTest else DAT.FT_train_out.shape[-1],
        d_model=32,
        n_heads=4,
        n_layers=5,
        act="gelu",
        block="mlp",
        norm="layer",
        dropout=0.1,
        pool="mean"
    ).to(device),
    lossf={
    "UT":CombinedCurveLoss(
        mse_weight=1.0,
        weighted_mse_weight=3.0,
        derivative_weight=0.5,
        peak_weight=3.0,
        energy_weight=1.0,
        peak_location_weight=2.0,
        zone_boundaries=(50, 125),
        zone_weights=(1.0, 5.0, 2.0),
        x_values=DAT.UT_OUT_df if DAT.UTmechTest else None,
        reduction="mean",
        derivative_order=1,
        SoftPeak_beta=20.0,)}, 
    # "FT":CombinedCurveLoss(
    #     mse_weight=0.1,
    #     weighted_mse_weight=1.0,
    #     derivative_weight=0.5,
    #     peak_weight=0.2,
    #     energy_weight=0.2,
    #     peak_location_weight=0.05,
    #     zone_boundaries=(90, 150),
    #     zone_weights=(1.0, 5.0, 2.0),
    #     x_values=DAT.FT_OUT_df if DAT.FTmechTest else None,
    #     reduction="mean",
    #     derivative_order=1,
    #     SoftPeak_beta=20.0)
    # },
    opt=("adam", 6.016e-6), #("adam", 1e-4),
    batch=32,
    lr=1e-3,
    data=DAT,
    mechMode=DAT.mechMode,
    scheduler=("min", 0.2427, 15, 1e-4), #("min", 0.2427, 18, 1e-4), 
    earlyStop=EarlyStopping(patience=18, min_delta=1e-4, verbose=True),
    w_init="auto",
    device=device,
    optTrial=None,
    scan_matches_on_init=True
)

TR1.summary()

In [ ]:
TR1.train(n_epochs=10, verbose=1, plot=True)

In [ ]:
TR1.save(path=None, name=None)

In [ ]:
TR1.predict(test_dataloader=None, plot=True)

In [ ]:
for i in range(len(TR1.UT_test_outputs)):
   plot_predictions(DAT.UT_OUT_df, TR1.UT_test_outputs, TR1.UT_truth, indx=i, d_out=False)

# END